## CUAD RAG — Scale-up baseline (Experiment N config, N=100)

Experiment N (HyDE + nomic + mistral-small3.2:24b + PROMPT_V2) was the best config found on N=20,
n_runs=3: Rel 0.540 / Util 0.366 / Comp 0.581 / Adh 30%. That N=20 result has too much sample
variance to draw firm conclusions from (see Retrieval_Strategy.md — "N=5 is too small" applies at
N=20 too, just less severely). This notebook re-runs the identical config at N=100, n_runs=1 to get
a stable baseline before trying anything new (per Retrieval_Strategy.md: for final/full-dataset runs,
drop n_runs to 1 — the dataset itself becomes the variance-reduction mechanism).

In [1]:
import os
import sys
import subprocess
import time

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")

if IN_COLAB or IN_KAGGLE:
    !pip install git+https://github.com/saikrishna1729/reliablerag.git@rag_pipeline/jithu datasets pandas -q
    !curl -fsSL https://ollama.com/install.sh | sh
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    !ollama pull nomic-embed-text-v2-moe:latest
    !ollama pull llama3.1:8b-instruct-q4_K_M
    !ollama pull mistral-small3.2:24b

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter

from reliablerag.chain import PROMPT_V2
from reliablerag.env import load_secrets
from reliablerag.experiment import evaluate_results, run_rag_experiment
from reliablerag.providers import create_embeddings, create_llm
from reliablerag.retriever import get_hyde_retriever

### 1. Configuration

Model names and paths are loaded from `.env`. Fallback defaults are used if not set.

In [4]:
load_secrets()

PROVIDER           = os.environ["PROVIDER"]
EMBEDDING_MODEL    = os.environ["EMBEDDING_MODEL"]
GENERATOR_MODEL    = os.environ["GENERATOR_MODEL"]
JUDGE_MODEL        = os.environ["JUDGE_MODEL"]
CHROMA_PERSIST_DIR = os.environ["CHROMA_PERSIST_DIR"]

print(f"Provider        : {PROVIDER}")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Generator model : {GENERATOR_MODEL}")
print(f"Judge model     : {JUDGE_MODEL}")
print(f"Chroma dir      : {CHROMA_PERSIST_DIR}")

Provider        : ollama
Embedding model : nomic-embed-text-v2-moe:latest
Generator model : mistral-small3.2:24b
Judge model     : llama3.1:8b-instruct-q4_K_M
Chroma dir      : /Users/jithamanyu.manne/git/others/python/reliablerag/data/chroma_db


In [5]:
embeddings = create_embeddings(PROVIDER, EMBEDDING_MODEL)

llm = create_llm(PROVIDER, "mistral-small3.2:24b")

judge_llm = create_llm(PROVIDER, JUDGE_MODEL, temperature=0)

hyde_llm = create_llm(PROVIDER, "llama3.1:8b-instruct-q4_K_M")

### 2. Load CUAD Samples from RAGBench

N=100 this time — large enough to average out the per-sample noise that made the N=20 numbers
hard to interpret.

In [6]:
N_SAMPLES = 100

dataset = load_dataset("galileo-ai/ragbench", "cuad", split="train")
samples = list(dataset.select(range(N_SAMPLES)))


def fmt(v):
    return f"{v:.3f}" if v is not None else "N/A"


print(f"Loaded {len(samples)} CUAD samples")

s = samples[0]
print(f"\nQuestion         : {s['question']}")
print(f"Doc length       : {len(s['documents'][0])} chars")
print(f"Adherence score  : {s['adherence_score']}")
print(f"Relevance score  : {fmt(s['relevance_score'])}")
print(f"Utilization score: {fmt(s['utilization_score'])}")
print(f"Completeness     : {fmt(s['completeness_score'])}")

Loaded 100 CUAD samples

Question         : Is one party required to deposit its source code into escrow with a third party, which can be released to the counterparty upon the occurrence of certain events (bankruptcy,  insolvency, etc.)?
Doc length       : 122054 chars
Adherence score  : True
Relevance score  : 0.000
Utilization score: 0.000
Completeness     : 1.000


### 3. Run Experiment N config (HyDE + nomic + mistral-small3.2:24b + PROMPT_V2) at N=100

In [7]:
# Experiment N @ N=100 — HyDE + nomic + PROMPT_V2 + mistral-small3.2:24b.
# Identical config to notebook 02's Experiment N; only N_SAMPLES changed (20 -> 100).
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}_n100"

results_n100 = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hyde_retriever(vs, embeddings, hyde_llm, top_k=TOP_K),
    embeddings=embeddings,
    generator_llm=llm,
    prompt_template=PROMPT_V2,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
    retrieve_label=f"hyde retrieve (top-{TOP_K})",
)


[1/100] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 5.337s  (291 chunks embedded)
[timing] hyde retrieve (top-20) : 5.970s
[timing] llm      : 1.276s
[timing] llm      : 27.492s
  our: Absent.
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/100] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 2.603s  (47 chunks embedded)
[timing] hyde retrieve (top-20) : 5.389s
[timing] llm      : 1.448s
[timing] llm      : 24.522s
  our: Absent.
  ref: No, the contract does not contain a license granted by one party to its counterparty. The contract is a Non-Competition Agreement and Right of First Offer between Glamis Gold Ltd. and Western Copper Corporation. It does not involve the granting of any lice

In [9]:
# Experiment N @ N=100 — TRACe evaluation. n_runs=1: dataset size is the variance-reduction
# mechanism at this scale (see Retrieval_Strategy.md "Notes for Scaling Up").
JUDGE_N_RUNS = 1
agg_n100 = evaluate_results(results_n100, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Exp N @ N=100 (HyDE + PROMPT_V2 + mistral-small3.2) — Rel {agg_n100['avg_relevance']:.3f} / Util {agg_n100['avg_utilization']:.3f} / Comp {agg_n100['avg_completeness']:.3f} / Adh {agg_n100['adherence_rate']:.0%}")

[1/100] [PASS] Is one party required to deposit its source code into escrow with a th...
  Adherence   : PASS  — The response as a whole is not supported by the documents. The question asks about deposit
  Relevance   : 0.334 — The relevant information for answering this question can be found in Section 13 of the doc
  Utilization : 0.046
  Completeness: 0.138

[_strip_trailing_commas] trailing comma stripped from judge output
[2/100] [FAIL] Does the contract contain a license granted by one party to its counte...
  Adherence   : FAIL  — The response as a whole is not supported by the documents. The response claims that there 
  Relevance   : 0.183 — The relevant information for answering this question can be found in PART 1 of the contrac
  Utilization : 0.000
  Completeness: 0.000

[3/100] [FAIL] The date when the contract is effective ...
  Adherence   : FAIL  — The response as a whole is not supported by the documents. The response claims that the co
  Relevance   : 0.051 — The doc